# Stage 1 Training Run Notebook

This notebook intentionally uses **one entry point only** for Stage 1:

```bash
python scripts/train_stage1.py
```

The small-part/high-resolution quality upgrade is now implemented inside the normal training script and config. There is no separate notebook training loop. Edit the paths in the shell commands below before running.

## 1. Check PartImageNet layout

This does not train. It only verifies that the dataset paths resolve.

In [5]:
!python scripts/train_stage1.py \
  --config configs/stage1_quality_upgrade.yaml \
  --partimagenet-root ../full_hyco/PartImageNet \
  --print-data-layout

PartImageNet layout check:
  root              : ../full_hyco/PartImageNet  [OK]
  train annotations : ../full_hyco/PartImageNet/annotations/train/train.json  [OK]
  val annotations   : ../full_hyco/PartImageNet/annotations/val/val.json  [OK]
  test annotations  : ../full_hyco/PartImageNet/annotations/test/test.json  [OK]
  train images      : ../full_hyco/PartImageNet/images/train  [OK]
  val images        : ../full_hyco/PartImageNet/images/val  [OK]
  test images       : ../full_hyco/PartImageNet/images/test  [OK]

Stage 1 uses annotations/<split>/<split>.json; *_whole folders are not used for part-mask training.


## 2. Smoke-test the exact Stage-1 setup

This runs one forward/audit pass with the same high-resolution refinement and quality-loss configuration used for training.

Use `--warm-start ... --allow-partial-load` when you are initializing the high-resolution model from an older Stage-1 checkpoint that did not contain the refinement layers.

In [6]:
!python scripts/train_stage1.py \
  --config configs/stage1_quality_upgrade.yaml \
  --device auto \
  --save-dir runs/stage1_quality_upgrade \
  --warm-start runs/stage1/checkpoints/stage1_best.pt \
  --allow-partial-load \
  --quality-loss \
  --use-highres-refine \
  --presence-threshold 0.25 \
  --topk-presence-k 128 \
  --small-part-area-tau 0.015 \
  --small-part-weight-max 6.0 \
  --small-part-weight-power 0.5 \
  --smoke-only

[dataset] train.json: 20457 samples | C=11 F=13 R=40
[dataset] val.json: 1205 samples | C=11 F=13 R=40
[datasets] train: ../full_hyco/PartImageNet/annotations/train/train.json | images: ../full_hyco/PartImageNet/images/train | samples: 20457
[datasets] val: ../full_hyco/PartImageNet/annotations/val/val.json | images: ../full_hyco/PartImageNet/images/val | samples: 1205
/home/dfli/anaconda3/envs/partviz/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/dfli/anaconda3/envs/partviz/lib/python3.12/

## 3. Full Stage-1 training

This is the only command needed to train the upgraded Stage 1. It writes:

```text
runs/stage1_quality_upgrade/checkpoints/stage1_best.pt
runs/stage1_quality_upgrade/checkpoints/stage1_last.pt
runs/stage1_quality_upgrade/stage1_history.csv
runs/stage1_quality_upgrade/stage1_history.json
```

In [8]:
!python scripts/train_stage1.py \
  --config configs/stage1_quality_upgrade.yaml \
  --device auto \
  --save-dir runs/stage1_quality_upgrade \
  --warm-start runs/stage1/checkpoints/stage1_best.pt \
  --allow-partial-load \
  --quality-loss \
  --use-highres-refine \
  --epochs 24 \
  --batch-size 8 \
  --lr 5e-5 \
  --presence-threshold 0.25 \
  --topk-presence-k 128 \
  --small-part-area-tau 0.015 \
  --small-part-weight-max 6.0 \
  --small-part-weight-power 0.5 \
  --quality-presence-bce 0.40 \
  --valid-absent-topmean-fp 0.08 \
  --valid-absent-mean-fp 0.02 \
  --invalid-part-topmean 0.35 \
  --invalid-part-mean 0.08 \
  --gt-support-leak 0.35 \
  --pred-support-containment 0.25 \
  --boundary-loss 0.08 \
  --focal-functional 0.12 \
  --tversky-functional 0.12 \
  --quality-topq 0.02

[dataset] train.json: 20457 samples | C=11 F=13 R=40
[dataset] val.json: 1205 samples | C=11 F=13 R=40
[datasets] train: ../full_hyco/PartImageNet/annotations/train/train.json | images: ../full_hyco/PartImageNet/images/train | samples: 20457
[datasets] val: ../full_hyco/PartImageNet/annotations/val/val.json | images: ../full_hyco/PartImageNet/images/val | samples: 1205
/home/dfli/anaconda3/envs/partviz/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/dfli/anaconda3/envs/partviz/lib/python3.12/

## 4. Optional: train from scratch

Use this only if no warm-start checkpoint exists. It still uses the same `train_stage1.py` entry point.

In [ ]:
# !python scripts/train_stage1.py \
#   --config configs/stage1_quality_upgrade.yaml \
#   --device auto \
#   --save-dir runs/stage1_quality_upgrade_from_scratch \
#   --quality-loss \
#   --use-highres-refine \
#   --epochs 24 \
#   --batch-size 8 \
#   --lr 5e-5

[dataset] train.json: 20457 samples | C=11 F=13 R=40
[dataset] val.json: 1205 samples | C=11 F=13 R=40
[datasets] train: ../full_hyco/PartImageNet/annotations/train/train.json | images: ../full_hyco/PartImageNet/images/train | samples: 20457
[datasets] val: ../full_hyco/PartImageNet/annotations/val/val.json | images: ../full_hyco/PartImageNet/images/val | samples: 1205
/home/dfli/anaconda3/envs/partviz/lib/python3.12/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/home/dfli/anaconda3/envs/partviz/lib/python3.12/